# Stage 3 — DINOv2 and DeiT-B on Oxford-IIIT Pet

Owner: Karim. This notebook runs the complete frozen-backbone retrieval experiment.

**Start:** select **Runtime → Change runtime type → T4 GPU**, upload the prepared `Visual_retrieval_system_stage3.zip` through the **Files** panel, then choose **Run all**. The ZIP includes the local implementation; the GitHub main branch alone does not yet include these edits.

Default: run **all four models in one GPU session** for a comparable runtime table. Quality uses every validation query (1,840) and every test query (3,669), against the same 1,840-image gallery. The 200-query sample is used only for latency. No training or fine-tuning takes place.

The official test split has already been evaluated in Stage 2. Keep the agreed settings fixed; choose any future settings using validation. Breed agreement is the relevance proxy. Compare the available pretrained backbones, accounting for architecture and pretraining-data differences.

Results, logs, source snapshot, embeddings and rankings are saved in `/content/icv_stage3_results/<run id>/` after each model; download the final results ZIP before closing the runtime. Real experimental results are produced only when these cells execute successfully.

In [ ]:
# Default protocol. Leave this list intact for a fresh four-model comparison.
MODELS = ["dino_vit_s16", "deit_vit_s16_supervised", "dinov2_vit_b14", "deit_vit_b16_supervised"]
TIMING_REPEATS = 3
BATCH_SIZE = 64
NUM_WORKERS = 2
EXPECTED_MANIFEST_SHA256 = "82f40e25888189aba46d1c1ff2c74894284c1ff7fa50362026c8abbea0ec9f80"
# Source ZIP uploaded through the Colab Files panel.
ZIP_PATH = ""

## Load the exact source snapshot

A fresh extraction folder prevents accidentally running an older Stage 2 ZIP. The notebook verifies every bundled file and the shared split manifest before execution.

In [ ]:
ZIP_PATH = "/content/Visual_retrieval_system_stage3.zip"
from google.colab import files
from pathlib import Path, PurePosixPath
import datetime as dt
import hashlib, json, os, shutil, subprocess, sys, uuid, zipfile

if not ZIP_PATH:
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith(".zip")]
    if len(candidates) != 1:
        raise ValueError("Upload exactly one Visual_retrieval_system_stage3.zip")
    ZIP_PATH = str(Path(candidates[0]).resolve())
    del uploaded
source_zip = Path(ZIP_PATH)
assert source_zip.is_file(), source_zip
work = Path("/content") / ("icv_stage3_" + uuid.uuid4().hex[:8])
work.mkdir()
with zipfile.ZipFile(source_zip) as archive:
    for name in archive.namelist():
        p = PurePosixPath(name)
        assert not p.is_absolute() and ".." not in p.parts, name
    archive.extractall(work)
REPO = work / "Visual_retrieval_system"
os.chdir(REPO)
bundle = json.loads((REPO / "bundle_manifest.json").read_text())
for relative, expected in bundle["files"].items():
    assert hashlib.sha256((REPO / relative).read_bytes()).hexdigest() == expected, relative
assert hashlib.sha256((REPO / "configs/split_manifest.csv").read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256
RUN_ID = "stage3_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN_ROOT = Path("/content/icv_stage3_results") / RUN_ID
for folder in ["logs", "evidence", "quality", "artifacts", "timing_repeats", "examples"]:
    (RUN_ROOT / folder).mkdir(parents=True, exist_ok=True)
shutil.copy2(source_zip, RUN_ROOT / "source_snapshot.zip")
(RUN_ROOT / "bundle_manifest.json").write_text(json.dumps(bundle, indent=2))
(RUN_ROOT / "run_settings.json").write_text(json.dumps({
    "models": MODELS, "timing_repeats": TIMING_REPEATS, "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS, "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "source_zip_sha256": hashlib.sha256(source_zip.read_bytes()).hexdigest(),
    "storage": "colab_vm_download_zip", "precision": "fp32", "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
}, indent=2))
print("Verified files:", len(bundle["files"]))
print("Source:", REPO)
print("VM results:", RUN_ROOT)

## Create an isolated Python environment

Use Colab's existing matching CUDA PyTorch/Torchvision pair. Install the tested retrieval dependencies into a separate virtual environment, so the notebook kernel does not need a restart. Record the actual versions, including CUDA and GPU.

In [ ]:
ENV = work / "venv"
subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(ENV)], check=True)
PYTHON = str(ENV / "bin/python")

def run(command, log_name):
    """Stream output and persist it; stop the notebook on a failed command."""
    log_path = RUN_ROOT / "logs" / (log_name + ".log")
    with log_path.open("a") as log:
        log.write("COMMAND " + repr(command) + "\n")
        log.flush()
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f"Command failed ({exit_code}); saved log: {log_path}")

run([PYTHON, "-m", "pip", "install", "-r", "requirements-stage3.txt"], "install")
freeze = subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True)
(RUN_ROOT / "evidence/environment.txt").write_text(freeze)
probe = "import json,torch,sys; sys.path.insert(0,'src'); from visual_retrieval import models,timing; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(json.dumps({'hardware':timing.hardware_info('cuda'),'registered_models':models.available_models()}))"
environment = json.loads(subprocess.check_output([PYTHON, "-c", probe], text=True, cwd=REPO))
assert set(MODELS).issubset(environment["registered_models"])
(RUN_ROOT / "evidence/hardware.json").write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))

## Preflight and the real-weight interface check

Check the existing split without rewriting it, run all tests, and verify the two new pretrained encoders through the shared top-5 retrieval interface. The synthetic smoke check does not measure Oxford-Pet quality.

In [ ]:
run([PYTHON, "scripts/prepare_data.py", "--check"], "dataset_check")
run([PYTHON, "-m", "pytest", "tests", "-q"], "tests")
run([PYTHON, "scripts/smoke_models.py", "--device", "cuda", "--output",
     str(RUN_ROOT / "evidence/pretrained_smoke.json")], "pretrained_smoke")

## Full quality evaluation

Each model embeds the complete gallery, validation and test sets. `--rebuild` forces fresh gallery timing. Results and rankings are saved to Drive after every model, separately from Stage 2's historical results. Gallery timing excludes saving arrays to disk.

In [ ]:
for number, name in enumerate(MODELS, 1):
    print(f"\nModel {number}/{len(MODELS)}: {name}")
    run([PYTHON, "scripts/evaluate.py", "--models", name, "--device", "cuda",
         "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
         "--artifacts", str(RUN_ROOT / "artifacts"),
         "--results-dir", str(RUN_ROOT / "quality"), "--rebuild"], "evaluate_" + name)
    result = json.loads((RUN_ROOT / "quality" / (name + ".json")).read_text())
    assert result["pretrained"] and not result["gallery_build_cache_reused"]
    assert result["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
    assert result["sizes"] == {"gallery": 1840, "val": 1840, "test": 3669}
    print(f"Saved {number}/{len(MODELS)} models to Colab VM")

## Repeat runtime measurements

Rebuild the full gallery and time the same 200 query IDs for each model in every repeat. Model order rotates between repeats. Query preprocessing, embedding and search remain separate; the summary reports median and min/max across repeats. These measurements do not change quality settings.

In [ ]:
run([PYTHON, "scripts/benchmark_timing.py", "--models", *MODELS, "--device", "cuda",
     "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
     "--repeats", str(TIMING_REPEATS), "--output-dir", str(RUN_ROOT / "timing_repeats")], "timing_repeats")

## Validate completeness and export examples

Require every expected metric/result file and an identical timing query sample. Export fixed validation examples and the first Hit@5 failure, if present. The saved rankings let the team analyze other failures later.

In [ ]:
quality_runs = [json.loads((RUN_ROOT / "quality" / (m + ".json")).read_text()) for m in MODELS]
expected_query_ids = quality_runs[0]["query_latency"]["query_ids"]
assert len(expected_query_ids) == 200 and len(set(expected_query_ids)) == 200
for result in quality_runs:
    assert result["query_latency"]["query_ids"] == expected_query_ids
    assert set(result["metrics"]) == {"val", "test"}
    for split in ["val", "test"]:
        assert set(result["metrics"][split]) == {"P@5", "P@10", "R@5", "R@10", "Hit@5"}
        assert all(0 <= value <= 1 for value in result["metrics"][split].values())
for name in MODELS:
    for repeat in range(1, TIMING_REPEATS + 1):
        record = json.loads((RUN_ROOT / "timing_repeats" / f"{name}_repeat{repeat}.json").read_text())
        assert record["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
        assert record["pretrained"] and not record["gallery_build_cache_reused"]
        assert record["query_latency"]["query_ids"] == expected_query_ids
        assert record["hardware"] == quality_runs[0]["hardware"]
# Validate complete array coverage, rather than trusting only aggregate JSON counts.
validator = """import sys,numpy as np; from pathlib import Path
sys.path.insert(0,'src'); from visual_retrieval import data
m=data.load_manifest('configs/split_manifest.csv')
base=Path(sys.argv[1])
for name in sys.argv[2:]:
 for split in ['gallery','val','test']:
  expected=data.subset(m,split).image_id.tolist()
  with np.load(base/'embeddings'/name/(split+'.npz'),allow_pickle=False) as z:
   assert z['image_ids'].tolist()==expected
   assert z['embeddings'].shape==(len(expected),768 if name in ['dinov2_vit_b14','deit_vit_b16_supervised'] else 384)
   assert np.isfinite(z['embeddings']).all()
   assert np.allclose(np.linalg.norm(z['embeddings'],axis=1),1,atol=1e-3)
 for split in ['val','test']:
  with np.load(base/'runs'/name/('topk_'+split+'.npz'),allow_pickle=False) as z:
   assert z['query_ids'].tolist()==data.subset(m,split).image_id.tolist()
   assert z['gallery_ids'].shape==(len(z['query_ids']),10)
print('PASS: complete embeddings and rankings for all requested models')
"""
run([PYTHON, "-c", validator, str(RUN_ROOT / "artifacts"), *MODELS], "coverage_check")
run([PYTHON, "scripts/export_examples.py", "--models", *MODELS,
     "--artifacts", str(RUN_ROOT / "artifacts"), "--output-dir", str(RUN_ROOT / "examples")], "examples")
from IPython.display import Markdown, Image, display
display(Markdown((RUN_ROOT / "quality/metrics.md").read_text()))
print("Repeated timings:")
print((RUN_ROOT / "timing_repeats/summary.csv").read_text())
for name in ["dinov2_vit_b14", "deit_vit_b16_supervised"]:
    image_path = RUN_ROOT / "examples" / (name + "_fixed.png")
    if image_path.exists():
        display(Image(filename=str(image_path)))

## Save and download the evidence bundle

The archive includes model results, repeated timings, logs, versions, embeddings, rankings, examples, the exact source ZIP and file hashes. Model weights and the dataset are excluded. Also save a copy of this **executed notebook** through **File → Download → Download .ipynb** to preserve its outputs for the report.

In [ ]:
status = {"status": "evaluation_complete", "models": MODELS, "n_models": len(MODELS),
          "gallery_images": 1840, "validation_queries": 1840, "test_queries": 3669,
          "timing_queries": 200, "timing_repeats_per_model": TIMING_REPEATS,
          "manifest_sha256": EXPECTED_MANIFEST_SHA256,
          "completed_utc": dt.datetime.now(dt.timezone.utc).isoformat()}
(RUN_ROOT / "run_status.json").write_text(json.dumps(status, indent=2))
content_hashes = {p.relative_to(RUN_ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                  for p in sorted(RUN_ROOT.rglob("*")) if p.is_file() and p.name != "file_hashes.json"}
(RUN_ROOT / "file_hashes.json").write_text(json.dumps(content_hashes, indent=2))
result_zip = RUN_ROOT.parent / (RUN_ID + "_results.zip")
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(RUN_ROOT.rglob("*")):
        if p.is_file():
            archive.write(p, RUN_ID + "/" + p.relative_to(RUN_ROOT).as_posix())
with zipfile.ZipFile(result_zip) as archive:
    assert archive.testzip() is None
    for relative, expected in content_hashes.items():
        assert hashlib.sha256(archive.read(RUN_ID + "/" + relative)).hexdigest() == expected
checksum = hashlib.sha256(result_zip.read_bytes()).hexdigest()
result_zip.with_suffix(".sha256").write_text(checksum + "  " + result_zip.name + "\n")
print(json.dumps(status, indent=2))
print("Saved on Colab VM:", result_zip)
print("SHA-256:", checksum)
files.download(str(result_zip))